# 第14章：软件流水线 -- 隐藏内存延迟

## 前置知识
- 第09章：分块矩阵乘法基础
- 第12章：Block Pointer 与 Shared Memory
- 第13章：Swizzle 与 L2 Cache 优化

## 学习目标
- 理解 **软件流水线 (Software Pipelining)** 的核心思想
- 理解 CUDA 中双缓冲 / 三缓冲的设计 (`simt_pipline.cu`, `mma_pipline.cu`)
- 掌握 Triton 中 `num_stages` 参数的作用
- 理解编译器如何生成异步拷贝指令 (`cp.async`)
- 实现不同 pipeline 深度的 GEMM 并对比性能

## 对应 CUDA 代码
- `src/simt/04simt_pipline.cu` — SIMT 双缓冲流水线 (0.723ms)
- `src/mma/04mma_pipline.cu` — MMA 双缓冲流水线 (0.411ms)
- 核心思想：加载下一个 tile 的同时计算当前 tile

In [ ]:
import torch
import triton
import triton.language as tl

## 14.1 内存延迟问题

### GPU 内存访问延迟

```
访问类型           延迟 (cycles)   带宽
────────────────────────────────────────────
Register           ~1              —
Shared Memory      ~20-30          ~TB/s
L2 Cache           ~200-300        ~2-4 TB/s
Global Memory      ~400-600        ~1-2 TB/s
```

### 无流水线的 GEMM (停顿问题)

```
时间线 (无流水线):

K=0:    |███ Load A0,B0 ███|██ Compute ██|                     ← 加载等待
K=BK:   |                  |             |███ Load A1,B1 ███|██ Compute ██|
K=2BK:  |                  |             |                  |             |███ Load ███|██ Compute ██|
                                                                          
        ▼ 计算单元空闲 ▼   ▼ 内存空闲 ▼  ▼ 计算单元空闲 ▼   ▼ 内存空闲 ▼

问题: 加载和计算串行执行
  - 加载时, 计算单元空闲 (ALU stall)
  - 计算时, 内存带宽浪费
  - 总时间 = 加载时间 + 计算时间 (没有重叠!)
```

### 流水线的解决方案

```
时间线 (双缓冲流水线):

加载:   |███ Load A0,B0 ███|███ Load A1,B1 ███|███ Load A2,B2 ███|███ Load A3,B3 ███|
计算:   |                  |██ Compute(A0,B0)██|██ Compute(A1,B1)██|██ Compute(A2,B2)██|█ Comp(A3,B3)█|
        
        ↑ 预加载第一个 tile  ↑ 加载和计算重叠!   ↑ 继续重叠         ↑ 最后只有计算

关键: 当计算 tile k 时, 同时加载 tile k+1
  - 加载和计算可以并行执行 (不同硬件单元)
  - 总时间 ≈ max(加载时间, 计算时间)  (理想情况)
  - 节省了大量等待时间!
```

这就是 CUDA 项目中 `simt_pipline.cu` 和 `mma_pipline.cu` 的核心优化思想。

## 14.2 CUDA Pipeline 的实现回顾

### simt_pipline.cu 的流水线结构

参考 CUDA 项目中的 ASCII 图：

```
 ←-----------------------------------------------------------------------------------
 ⤷---------------------------------------iter k-----------------------------------→-⤴
 |████████████████load global███████████████████████|███store shared███|             |  Global to Shared Memory
 |---------------------------------------iter bk-----------------------↘-------------|  
 |█load shared█|█load shared█|█load shared█|█load shared█|█load shared█|█load shared█|  Shared Memory to Registers
 ↘-------------↘------------↘-------------↘-------------↘-------------↘-------------↘
 |████Math█████|████Math█████|████Math█████|████Math█████|████Math█████|████Math█████|  Registers to CUDA cores
```

### 三层流水线

CUDA 实现了三层独立的流水线：

1. **Global → Shared Memory**：异步加载下一个 K tile
2. **Shared → Register**：加载下一个 bk 子步的数据
3. **Register → Compute**：执行乘加运算

### 双缓冲 (Double Buffering) 的关键

```
CUDA 中的双缓冲:

__shared__ float4 smem_A[2][...];   // 两个 buffer
__shared__ float4 smem_B[2][...];
float4 reg_a[2][...];               // 两个寄存器组
float4 reg_b[2][...];

// 交替使用:
smem_write_idx = 0;  // 写入 buffer 0
smem_read_idx = 0;   // 从 buffer 0 读取
...
smem_write_idx = !smem_write_idx;  // 切换到 buffer 1
// 写入 buffer 1 的同时, 从 buffer 0 读取 → 重叠!
```

```
双缓冲示意图:

时间:    T0        T1        T2        T3        T4

Buffer0: [写入A0]  [读取A0]  [写入A2]  [读取A2]  [写入A4]
Buffer1: [空闲]    [写入A1]  [读取A1]  [写入A3]  [读取A3]

         写和读交替使用不同的 buffer → 不冲突!
```

## 14.3 Triton 的流水线：num_stages 参数

### Triton 的简洁方案

在 CUDA 中需要手动管理双缓冲、手动交替读写、手动插入异步拷贝指令...
在 Triton 中，**只需要一个参数**：

```python
@triton.jit
def kernel(..., num_stages: tl.constexpr):
    for k in range(0, K, BLOCK_K):
        a = tl.load(a_ptr, ...)   # 编译器自动流水线化
        b = tl.load(b_ptr, ...)   # 编译器自动流水线化
        acc = tl.dot(a, b, acc=acc)
```

通过 kernel launch 时指定 `num_stages`：

```python
kernel[grid](..., num_stages=3)  # 三级流水线
```

### num_stages 的含义

```
num_stages = 1: 无流水线
  加载 → 计算 → 加载 → 计算 → ...
  Shared Memory 使用: 1 * (BM*BK + BK*BN) * sizeof(half)

num_stages = 2: 双缓冲 (对应 CUDA 的 smem_A[2])
  加载 tile 1 → 同时 {加载 tile 2, 计算 tile 1} → ...
  Shared Memory 使用: 2 * (BM*BK + BK*BN) * sizeof(half)

num_stages = 3: 三缓冲
  预加载 tile 1,2 → 同时 {加载 tile 3, 计算 tile 1} → ...
  Shared Memory 使用: 3 * (BM*BK + BK*BN) * sizeof(half)

num_stages = 4: 四缓冲
  预加载 tile 1,2,3 → 同时 {加载 tile 4, 计算 tile 1} → ...
  Shared Memory 使用: 4 * (BM*BK + BK*BN) * sizeof(half)
```

### 编译器生成的指令

```
num_stages > 1 时, 编译器会在 Ampere+ GPU 上生成:

1. cp.async.ca.shared.global  (异步全局→共享内存拷贝)
   - 不阻塞计算管线
   - 硬件 DMA 引擎执行数据搬运

2. cp.async.commit_group      (提交一组异步拷贝)

3. cp.async.wait_group N      (等待第 N 组之前的拷贝完成)
   - N=0: 等待所有拷贝完成
   - N=1: 允许最后1组拷贝还在进行中

这些指令在 Turing 之前的 GPU 上不可用, 编译器会退化为同步拷贝。
```

## 14.4 实现：不同 Pipeline 深度的 GEMM

In [ ]:
@triton.jit
def matmul_pipeline_kernel(
    a_ptr, b_ptr, c_ptr,
    M, N, K,
    stride_am, stride_ak,
    stride_bk, stride_bn,
    stride_cm, stride_cn,
    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
    BLOCK_K: tl.constexpr,
    GROUP_SIZE_M: tl.constexpr,
):
    """
    带 Swizzle 的 GEMM kernel, 支持通过 num_stages 控制流水线深度。
    
    与 CUDA simt_pipline.cu / mma_pipline.cu 的核心区别:
    - CUDA 需要手动管理双缓冲: smem_A[2][...], reg_a[2][...]
    - CUDA 需要手动交替 read/write index: smem_write_idx = !smem_write_idx
    - CUDA 需要手动处理 prologue (首次加载) 和 epilogue (最后计算)
    - Triton 只需要写正常的循环, 编译器自动插入流水线逻辑!
    """
    # Swizzled pid mapping (来自第13章)
    pid = tl.program_id(0)
    grid_m = tl.cdiv(M, BLOCK_M)
    grid_n = tl.cdiv(N, BLOCK_N)
    group_id = pid // (GROUP_SIZE_M * grid_n)
    first_pid_m = group_id * GROUP_SIZE_M
    group_size_m = min(grid_m - first_pid_m, GROUP_SIZE_M)
    pid_m = first_pid_m + ((pid % (GROUP_SIZE_M * grid_n)) % group_size_m)
    pid_n = (pid % (GROUP_SIZE_M * grid_n)) // group_size_m
    
    # Block Pointer
    a_block_ptr = tl.make_block_ptr(
        base=a_ptr, shape=(M, K), strides=(stride_am, stride_ak),
        offsets=(pid_m * BLOCK_M, 0), block_shape=(BLOCK_M, BLOCK_K), order=(1, 0),
    )
    b_block_ptr = tl.make_block_ptr(
        base=b_ptr, shape=(K, N), strides=(stride_bk, stride_bn),
        offsets=(0, pid_n * BLOCK_N), block_shape=(BLOCK_K, BLOCK_N), order=(1, 0),
    )
    
    # 累加器
    acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)
    
    # K 循环 — 编译器根据 num_stages 自动生成流水线代码
    # 对应 CUDA:
    #   LOAD_GLOBAL(0); STORE_SHARED(0); __syncthreads();  // prologue
    #   LOAD_SHARED(0, 0, 0);                               // 首次 smem → reg
    #   for (k = 1; k < K/BK; k++) { ... }                  // main loop with overlap
    #   // epilogue: 处理最后一个 tile
    for k in range(0, K, BLOCK_K):
        a_tile = tl.load(a_block_ptr, boundary_check=(0, 1))
        b_tile = tl.load(b_block_ptr, boundary_check=(0, 1))
        acc = tl.dot(a_tile, b_tile, acc=acc)
        a_block_ptr = tl.advance(a_block_ptr, (0, BLOCK_K))
        b_block_ptr = tl.advance(b_block_ptr, (BLOCK_K, 0))
    
    # 写回
    c_block_ptr = tl.make_block_ptr(
        base=c_ptr, shape=(M, N), strides=(stride_cm, stride_cn),
        offsets=(pid_m * BLOCK_M, pid_n * BLOCK_N),
        block_shape=(BLOCK_M, BLOCK_N), order=(1, 0),
    )
    tl.store(c_block_ptr, acc.to(tl.float16), boundary_check=(0, 1))

In [ ]:
def matmul_pipeline(a, b, BLOCK_M=128, BLOCK_N=128, BLOCK_K=32,
                    GROUP_SIZE_M=8, num_stages=2):
    """带流水线控制的 GEMM。"""
    M, K = a.shape
    K2, N = b.shape
    assert K == K2
    c = torch.empty((M, N), device=a.device, dtype=a.dtype)
    grid = (triton.cdiv(M, BLOCK_M) * triton.cdiv(N, BLOCK_N),)
    
    matmul_pipeline_kernel[grid](
        a, b, c, M, N, K,
        a.stride(0), a.stride(1), b.stride(0), b.stride(1),
        c.stride(0), c.stride(1),
        BLOCK_M=BLOCK_M, BLOCK_N=BLOCK_N, BLOCK_K=BLOCK_K,
        GROUP_SIZE_M=GROUP_SIZE_M,
        num_stages=num_stages,
    )
    return c

In [ ]:
# ========== 正确性验证 ==========
torch.manual_seed(42)
M, N, K = 2048, 2048, 1024
a = torch.randn(M, K, device='cuda', dtype=torch.float16)
b = torch.randn(K, N, device='cuda', dtype=torch.float16)
c_ref = torch.matmul(a, b)

print("不同 num_stages 的正确性验证:")
for stages in [1, 2, 3, 4]:
    c_tri = matmul_pipeline(a, b, num_stages=stages)
    max_err = (c_tri - c_ref).abs().max().item()
    print(f"  num_stages={stages}: max_err={max_err:.4f}, "
          f"correct={torch.allclose(c_tri, c_ref, atol=1.0)}")

## 14.5 双缓冲 vs 三缓冲

### 时间线对比

```
双缓冲 (num_stages=2):

Stage: ┌────Load────┐┌────Load────┐┌────Load────┐┌────Load────┐
       │   tile 0   ││   tile 1   ││   tile 2   ││   tile 3   │
       └────────────┘└────────────┘└────────────┘└────────────┘
                     ┌──Compute──┐ ┌──Compute──┐ ┌──Compute──┐ ┌──Compute──┐
                     │  tile 0   │ │  tile 1   │ │  tile 2   │ │  tile 3   │
                     └───────────┘ └───────────┘ └───────────┘ └───────────┘
       ↑ prologue     ↑ 重叠开始    ↑ 稳态        ↑ 稳态        ↑ epilogue

问题: 如果 Load 时间 > Compute 时间, 计算单元仍有空闲


三缓冲 (num_stages=3):

Stage: ┌──Load──┐┌──Load──┐┌──Load──┐┌──Load──┐┌──Load──┐
       │ tile 0 ││ tile 1 ││ tile 2 ││ tile 3 ││ tile 4 │
       └────────┘└────────┘└────────┘└────────┘└────────┘
                          ┌─Compute─┐┌─Compute─┐┌─Compute─┐┌─Compute─┐┌─Compute─┐
                          │ tile 0  ││ tile 1  ││ tile 2  ││ tile 3  ││ tile 4  │
                          └─────────┘└─────────┘└─────────┘└─────────┘└─────────┘
       ↑ 预加载2个         ↑ 加载 tile 2 时, tile 0 和 1 都已就绪
                            → 更好地隐藏延迟!

优势: 当加载延迟波动时, 有更多的 buffer 来吸收波动
代价: 需要更多 Shared Memory (3倍 vs 2倍)
```

### Shared Memory 使用量

```
以 BLOCK_M=128, BLOCK_N=128, BLOCK_K=32, FP16 为例:

每个 stage 需要:
  A tile: 128 * 32 * 2 bytes = 8 KB
  B tile: 32 * 128 * 2 bytes = 8 KB
  每 stage: 16 KB

num_stages=1: 16 KB  (可能在 L1 / smem 中)
num_stages=2: 32 KB  (大多数 GPU 都够用)
num_stages=3: 48 KB  (接近 SM 的 smem 上限)
num_stages=4: 64 KB  (可能超出某些 GPU 的限制!)

注意: 如果 Shared Memory 超出限制, kernel 会启动失败
或者 Triton 编译器会自动降低 num_stages
```

In [ ]:
# ========== 不同 num_stages 的性能对比 ==========
def benchmark_pipeline(M, N, K, BLOCK_M, BLOCK_N, BLOCK_K, num_stages,
                       num_warmup=10, num_rep=50):
    a = torch.randn(M, K, device='cuda', dtype=torch.float16)
    b = torch.randn(K, N, device='cuda', dtype=torch.float16)
    
    for _ in range(num_warmup):
        matmul_pipeline(a, b, BLOCK_M, BLOCK_N, BLOCK_K, num_stages=num_stages)
    torch.cuda.synchronize()
    
    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    start.record()
    for _ in range(num_rep):
        matmul_pipeline(a, b, BLOCK_M, BLOCK_N, BLOCK_K, num_stages=num_stages)
    end.record()
    torch.cuda.synchronize()
    
    ms = start.elapsed_time(end) / num_rep
    tflops = 2.0 * M * N * K / (ms * 1e-3) / 1e12
    return ms, tflops

M, N, K = 2048, 2048, 1024
BM, BN, BK = 128, 128, 32

print(f"矩阵规模: M={M}, N={N}, K={K}")
print(f"Block: ({BM}, {BN}, {BK})")
print(f"每 stage SMEM 使用: {(BM*BK + BK*BN)*2/1024:.1f} KB")
print(f"\n{'num_stages':>12} | {'SMEM(KB)':>10} | {'时间(ms)':>10} | {'TFLOPS':>8}")
print("-" * 55)

for stages in [1, 2, 3, 4, 5]:
    smem_kb = stages * (BM * BK + BK * BN) * 2 / 1024
    try:
        ms, tflops = benchmark_pipeline(M, N, K, BM, BN, BK, stages)
        print(f"{stages:>12} | {smem_kb:>10.1f} | {ms:>10.3f} | {tflops:>8.2f}")
    except Exception as e:
        print(f"{stages:>12} | {smem_kb:>10.1f} | {'FAIL':>10} | {str(e)[:30]}")

In [ ]:
# ========== 不同矩阵尺寸下的流水线效果 ==========
print("不同矩阵尺寸下 num_stages 的影响")
print(f"{'Size':>20} | {'stage=1':>10} {'stage=2':>10} {'stage=3':>10} | {'最优stage':>10}")
print("-" * 75)

for M, N, K in [
    (512, 512, 512),
    (1024, 1024, 1024),
    (2048, 2048, 1024),
    (2048, 2048, 2048),
    (4096, 4096, 2048),
]:
    results = []
    for stages in [1, 2, 3]:
        try:
            ms, _ = benchmark_pipeline(M, N, K, 128, 128, 32, stages)
            results.append((stages, ms))
        except:
            results.append((stages, float('inf')))
    
    best = min(results, key=lambda x: x[1])
    size_str = f"{M}x{N}x{K}"
    ms_strs = [f"{r[1]:>9.3f}ms" if r[1] < float('inf') else f"{'FAIL':>10}" for r in results]
    print(f"{size_str:>20} | {ms_strs[0]} {ms_strs[1]} {ms_strs[2]} | {'stage='+str(best[0]):>10}")

## 14.6 与 CUDA Pipeline 代码的详细对比

### CUDA simt_pipline.cu 的代码量

```cpp
// CUDA: 需要 4 个宏 + 复杂的主循环 (~220 行)

// 宏1: LOAD_GLOBAL — 从 global memory 加载到 register buffer
#define LOAD_GLOBAL(_K) { ... ~10行 ... }

// 宏2: STORE_SHARED — 从 register buffer 存到 shared memory
#define STORE_SHARED(WRITE_IDX) { ... ~12行 ... }

// 宏3: LOAD_SHARED — 从 shared memory 加载到计算寄存器
#define LOAD_SHARED(BK, SMEM_READ_IDX, REG_WRITE_IDX) { ... ~8行 ... }

// 主循环: 三层流水线交织
// Prologue:
LOAD_GLOBAL(0);           // 预加载第一个 tile
STORE_SHARED(0);          // 存到 smem buffer 0
__syncthreads();
LOAD_SHARED(0, 0, 0);    // 预加载第一个 bk

// Main loop:
for (int k = 1; k < K/BlockTileK; k++) {
    smem_write_idx = !smem_write_idx;
    LOAD_GLOBAL(k);       // 异步加载下一个 tile
    
    for (int bk = 1; bk < BlockTileK; bk++) {
        reg_write_idx = !reg_write_idx;
        LOAD_SHARED(bk, !smem_write_idx, reg_write_idx);  // 从另一个 buffer 读
        // COMPUTE with !reg_write_idx                      // 用之前的寄存器计算
    }
    
    // 最后一个 bk: 特殊处理, 重叠 STORE_SHARED 和 COMPUTE
    STORE_SHARED(smem_write_idx);
    // COMPUTE
    __syncthreads();
    LOAD_SHARED(0, smem_write_idx, !reg_write_idx);
    reg_write_idx = !reg_write_idx;
}

// Epilogue:
for (int bk = 0; bk < BlockTileK; bk++) {
    // ... 处理最后一个 tile
}
```

### Triton 的等效代码

```python
# Triton: 整个流水线只有 ~5 行!
for k in range(0, K, BLOCK_K):
    a_tile = tl.load(a_block_ptr, boundary_check=(0, 1))
    b_tile = tl.load(b_block_ptr, boundary_check=(0, 1))
    acc = tl.dot(a_tile, b_tile, acc=acc)
    a_block_ptr = tl.advance(a_block_ptr, (0, BLOCK_K))
    b_block_ptr = tl.advance(b_block_ptr, (BLOCK_K, 0))
# + num_stages=3 在 launch 时指定
```

| 方面 | CUDA (simt_pipline) | Triton |
|------|--------------------|---------|
| 代码行数 | ~220 行 | ~10 行 |
| 缓冲管理 | 手动 `smem[2]`, `reg[2]` | 编译器自动 |
| 索引切换 | `idx = !idx` | 编译器自动 |
| Prologue/Epilogue | 手动特殊处理 | 编译器自动 |
| 异步拷贝 | 手动 `cp.async` (如果使用) | 编译器自动 |
| 流水线深度 | 硬编码 (双缓冲) | 运行时可调 (num_stages) |

In [ ]:
# ========== BLOCK_K 与 num_stages 的交互效应 ==========
M, N, K = 2048, 2048, 2048

print(f"BLOCK_K 与 num_stages 的交互 (M={M}, N={N}, K={K})")
print(f"{'BLOCK_K':>8} {'stages':>8} | {'SMEM(KB)':>10} {'时间(ms)':>10} {'TFLOPS':>8}")
print("-" * 55)

for bk in [16, 32, 64]:
    for stages in [1, 2, 3]:
        smem_kb = stages * (128 * bk + bk * 128) * 2 / 1024
        try:
            ms, tflops = benchmark_pipeline(M, N, K, 128, 128, bk, stages)
            print(f"{bk:>8} {stages:>8} | {smem_kb:>10.1f} {ms:>10.3f} {tflops:>8.2f}")
        except Exception as e:
            print(f"{bk:>8} {stages:>8} | {smem_kb:>10.1f} {'FAIL':>10} {str(e)[:20]}")

## 14.7 总结

### 本章要点

1. **软件流水线的核心思想**：当计算当前 tile 时，同时加载下一个 tile，重叠延迟

2. **CUDA vs Triton**：
   - CUDA 需要手动实现双缓冲（`smem[2]`, `reg[2]`），手动管理索引切换，手动处理 prologue/epilogue
   - Triton 只需指定 `num_stages` 参数，编译器自动完成所有工作

3. **num_stages 的选择**：
   - `1`: 无流水线，最少 SMEM
   - `2`: 双缓冲，最常用
   - `3-4`: 更深的流水线，可能更好地隐藏延迟，但需要更多 SMEM
   - 受 Shared Memory 容量限制

4. **异步拷贝 (cp.async)**：
   - Ampere+ GPU 支持硬件异步 global→shared 拷贝
   - Triton 编译器自动生成这些指令
   - 进一步减少流水线的同步开销

5. **性能影响因素**：
   - 矩阵大小：大矩阵受益更多
   - BLOCK_K：更大的 BLOCK_K 减少循环次数但增加 SMEM 使用
   - GPU 架构：Ampere+ 的异步拷贝效果更好

### 练习

1. **SMEM 上限实验**：增大 BLOCK_M/BLOCK_N 直到 num_stages=2 也无法编译，观察报错信息
2. **K 维度影响**：固定 M=N=2048，改变 K (256, 512, 1024, 2048, 4096)，观察不同 num_stages 的效果
3. **PTX 分析**：查看 num_stages=1 和 num_stages=3 的 PTX 差异，寻找 `cp.async` 指令
4. **思考题**：为什么在 K 很小时 (如 K=64)，流水线可能反而更慢？（提示：prologue 开销）

### 下一章预告

第15章将介绍 **Split-K 并行**，通过在 K 维度引入并行性来加速 tall-skinny 矩阵乘法。